# Vv-Gamma チャート作成例

`VVGammaChart.py` の薄い使用例です。主計算はモジュール側に置き、notebook は条件設定・実行・可視化だけにします。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
import os

# ../bin/AnalysisVSPAERO.py をモジュールとしてインポート
sys.path.append(os.path.join('../..')) # 親ディレクトリをモジュール探索パスに追加
from src.VVGammaChart import (
    calculate_vv_gamma_metrics_from_stab,
    plot_vv_gamma_contour,
    run_vv_gamma_chart,
)

NameError: name 'os' is not defined

## 1. 既存 `.stab` の読み取り確認

OpenVSP が使えない環境でも、`.stab` の後処理だけは確認できます。

In [ ]:
metrics = calculate_vv_gamma_metrics_from_stab(
    'G103A.stab',
    vv=np.nan,
    tip_deflection=np.nan,
)
pd.Series(metrics)[[
    'Gamma_eff_deg',
    'CL_alpha',
    'Cl_beta',
    'Cn_beta',
    'Cl_delta_r',
    'Cn_delta_r',
    'spiral_margin',
    'control_column_delta_r',
]]

## 2. Vv-Gamma sweep の条件

以下は OpenVSP Python API が使える環境で実行します。`turn_trim_mode` は `none`, `level`, `gliding` から選べます。

In [ ]:
base_vsp3_path = Path('G103A.vsp3')
output_dir = Path('vv_gamma_cases')

vv_values = np.linspace(0.02, 0.08, 4)
tip_deflections = np.linspace(0.0, 0.5, 4)

flight_condition = {
    'alpha_deg': 2.0,
    'mach': 0.1,
    'reynolds': 4.4e6,
}

geometry_config = {
    'lv': 4.0,          # x_ac_vtail - xcg, model length unit
    'wing_area': 17.8819,
    'wing_span': 17.5366,
    'xcg': 2.87,
    'wing_name': 'WingGeom',
    'vtail_name': 'VTailGeom',
    'n_span': 101,
}

In [ ]:
results = run_vv_gamma_chart(
    base_vsp3_path,
    vv_values,
    tip_deflections,
    flight_condition,
    geometry_config,
    output_dir,
    turn_trim_mode='none',
    validate_base_model=True,
    verbose=1,
)
results.head()

## 3. 可視化

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
plot_vv_gamma_contour(results, 'spiral_margin', ax=ax)
plt.show()

## 4. turn trim 指標を入れる場合

滑空旋回を評価する場合は `turn_trim_mode='gliding'` を使います。`fixed` は既存 solver の仕様に合わせて3つだけ指定します。

In [ ]:
# results_with_trim = run_vv_gamma_chart(
#     base_vsp3_path,
#     vv_values,
#     tip_deflections,
#     flight_condition,
#     geometry_config,
#     output_dir / 'with_trim',
#     turn_trim_mode='gliding',
#     turn_trim_fixed={'V': 30.0, 'Omega': 0.08, 'delta_a': 0.0},
#     mass=600.0,
#     validate_base_model=True,
#     verbose=1,
# )